# AIoT Parte 3 — Lezione 01
# Il giro completo: da Colab ad Arduino

## Da dove viene questo notebook, e cosa è cambiato

Questo **è il notebook della sinusoide che avete già usato e commentato nella Parte 1** del corso:
stessi dati, stessa rete, stessi nomi delle variabili. Lo riprendiamo di proposito, perché così
l'unica cosa nuova da imparare qui è **l'ultimo tratto di strada**: come si porta quel modello
dentro un microcontrollore.

Rispetto alla Parte 1 ho cambiato quattro cose, e **nessuna di queste è un capriccio o un
miglioramento estetico**: sono tutte imposte dal fatto che TensorFlow è cambiato dal giorno in cui
quel notebook è stato scritto, e che questa volta il modello non deve girare su un computer ma
dentro un chip. Le elenco qui in modo che sappiate sempre cosa state guardando; ognuna è poi
spiegata per esteso nel punto in cui compare.

| # | Cosa cambia | Perché |
|---|---|---|
| 1 | `layers.Input(shape=(1,))` al posto di `input_shape=(1,)` dentro `Dense` | Forma superata: le versioni recenti di Keras vogliono che la forma dell'ingresso sia dichiarata da sola, come primo strato. |
| 2 | La conversione diventa **interamente a numeri interi**: tre righe in più | Nella Parte 1 il modello convertito accettava e restituiva numeri con la virgola. Va bene per un numero solo, ma nella Lezione 02 l'ingresso saranno 16.000 numeri: in virgola mobile occuperebbero 64 KB di RAM, un quarto di tutta la memoria della scheda. La ricetta dev'essere a numeri interi fin da subito, o le lezioni ne insegnerebbero due diverse. |
| 3 | Una riga che disattiva la quantizzazione "per canale" sugli strati Dense | È la correzione più importante e la meno intuitiva. Spiegata per esteso al Passo 6. Senza di essa il modello funziona sul computer e dà numeri sbagliati sulla scheda, **senza segnalare alcun errore**. |
| 4 | L'array C si genera qui dentro, con poche righe di Python, invece che con `xxd` | `xxd` richiede un comando di installazione, e le alternative su Windows richiedono PowerShell. Poche righe di Python funzionano su qualunque computer e ci permettono di dare all'array direttamente il nome che serve allo sketch. |

Una cosa che invece **non** è cambiata: la struttura in 8 passaggi. È la stessa che ritroverete
in ogni progetto di questo libro.

| | Passaggio | Dove si fa |
|---|---|---|
| 1 | Raccogliere i dati | qui su Colab |
| 2 | Preparare i dati | qui su Colab |
| 3 | Progettare il modello | qui su Colab |
| 4 | Addestrare | qui su Colab |
| 5 | Valutare | qui su Colab |
| 6 | Convertire e comprimere | qui su Colab |
| 7 | Scrivere il codice di distribuzione | nell'IDE Arduino |
| 8 | Distribuire l'applicazione | sulla scheda |

In [ ]:
# TensorFlow è la libreria di apprendimento automatico
import tensorflow as tf
# NumPy è la libreria di calcolo numerico
import numpy as np
# Matplotlib serve a disegnare i grafici
import matplotlib.pyplot as plt
# math è la libreria matematica di Python
import math

# Quanti punti generiamo
SAMPLES = 1000

# Fissiamo il "seme" del caso, così i numeri casuali sono gli stessi a ogni esecuzione
# e voi vedrete gli stessi risultati stampati in questo libro.
SEED = 1337
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow versione:", tf.__version__)

## Passo 1 — Raccogliere i dati

Mille numeri presi a caso fra 0 e 2π (un giro completo), e per ognuno il suo seno: sono le
domande e le risposte giuste su cui la rete si allenerà. Nella Parte 2 questo passaggio voleva
dire scaricare migliaia di registrazioni audio; qui bastano due righe.

In [ ]:
# Numeri casuali distribuiti uniformemente fra 0 e 2π, cioè un'oscillazione completa
x_values = np.random.uniform(low=0, high=2 * math.pi, size=SAMPLES)

# Li mescoliamo, per essere sicuri che non siano in ordine
np.random.shuffle(x_values)

# Calcoliamo il seno corrispondente
y_values = np.sin(x_values)

# Disegniamo i dati. 'b.' vuol dire "punti blu".
plt.plot(x_values, y_values, 'b.')
plt.title("I dati puliti: per ogni x, il suo seno")
plt.show()

## Passo 2 — Preparare i dati

Prima aggiungiamo un po' di **rumore** alle risposte. Sembra un'assurdità — stiamo sporcando
apposta dei dati perfetti — ma serve: nel mondo vero nessun sensore è preciso, e una rete che ha
visto solo dati impeccabili va in crisi al primo microfono un po' scadente.

Poi dividiamo i dati in tre gruppi: il 60% per allenare, il 20% per controllare i progressi
durante l'allenamento, e un ultimo 20% messo da parte e mai mostrato alla rete, che useremo alla
fine per darle un voto onesto. È come studiare sugli esercizi del libro, poi fare le simulazioni,
e infine affrontare il compito in classe con esercizi mai visti.

> **Piccola differenza dalla Parte 1.** Là la riga del rumore era `y_values += 0.1 * ...`, cioè
> *aggiungeva* rumore a quello che già c'era. Se rieseguite quella cella due o tre volte — cosa
> che su Colab capita in continuazione — il rumore si somma a ogni giro e i dati peggiorano senza
> che ve ne accorgiate. Qui il seno viene ricalcolato da capo ogni volta, così la cella si può
> rilanciare quante volte volete. È una buona abitudine da tenere per qualunque notebook.

In [ ]:
# Aggiungiamo a ogni risposta un piccolo numero casuale.
# Nota: ripartiamo da np.sin(x_values), non da y_values, così rilanciare questa cella
# non accumula rumore sopra rumore.
y_values = np.sin(x_values) + 0.1 * np.random.randn(*x_values.shape)

plt.plot(x_values, y_values, 'r.')
plt.title("Gli stessi dati, con il rumore che c'e' sempre nel mondo vero")
plt.show()

In [ ]:
# Il 60% per allenare, il 20% per il test, il restante 20% per la validazione.
TRAIN_SPLIT = int(0.6 * SAMPLES)
TEST_SPLIT = int(0.2 * SAMPLES + TRAIN_SPLIT)

# np.split taglia i dati nei punti indicati: due indici, quindi tre pezzi.
x_train, x_test, x_validate = np.split(x_values, [TRAIN_SPLIT, TEST_SPLIT])
y_train, y_test, y_validate = np.split(y_values, [TRAIN_SPLIT, TEST_SPLIT])

# Controllo: i tre pezzi devono ricomporre il totale
assert (x_train.size + x_test.size + x_validate.size) == SAMPLES

plt.plot(x_train, y_train, 'b.', label="Allenamento")
plt.plot(x_test, y_test, 'r.', label="Test")
plt.plot(x_validate, y_validate, 'y.', label="Validazione")
plt.legend()
plt.show()

## Passo 3 — Progettare il modello

La stessa rete della Parte 1: un numero entra, attraversa tre strati da 16 neuroni ciascuno, e un
numero esce. In tutto **593 parametri**. Per fare un paragone, il modello YAMNet che avete usato
nella Parte 2 ne ha diversi milioni.

> **Modifica 1 — perché.** Nella Parte 1 il primo strato era scritto così:
> `layers.Dense(16, activation='relu', input_shape=(1,))`. Quella forma funziona ancora, ma le
> versioni recenti di Keras la considerano superata e preferiscono che la forma dell'ingresso sia
> dichiarata da sola, come primo elemento della lista. Il modello che ne esce è identico: cambia
> solo il modo di scriverlo. Prendiamo l'abitudine nuova perché nella Lezione 02, dove l'ingresso
> è molto più complicato di un numero solo, dichiararlo bene fin dall'inizio farà una differenza
> concreta sulla memoria occupata.

In [ ]:
from tensorflow.keras import layers

model_2 = tf.keras.Sequential()

# La forma dell'ingresso, dichiarata per prima: un solo numero.
model_2.add(layers.Input(shape=(1,)))

# Primo strato: 16 "neuroni". Decidono se attivarsi in base alla funzione 'relu'.
model_2.add(layers.Dense(16, activation='relu'))
# Secondo strato
model_2.add(layers.Dense(16, activation='relu'))
# Terzo strato
model_2.add(layers.Dense(16, activation='relu'))
# Ultimo strato: un solo neurone, perché vogliamo in uscita un solo valore
model_2.add(layers.Dense(1))

model_2.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])
model_2.summary()

## Passo 4 — Addestrare

Seicento ripassi sugli stessi esercizi. Su Colab dura una manciata di secondi.

In [ ]:
history_2 = model_2.fit(x_train, y_train, epochs=600, batch_size=16,
                        validation_data=(x_validate, y_validate), verbose=0)

# Escludiamo le prime epoche, altrimenti il grafico e' schiacciato dai valori iniziali
SKIP = 80
loss = history_2.history['loss']
val_loss = history_2.history['val_loss']
epochs = range(1, len(loss) + 1)

plt.plot(epochs[SKIP:], loss[SKIP:], 'g.', label='Errore in allenamento')
plt.plot(epochs[SKIP:], val_loss[SKIP:], 'b.', label='Errore in validazione')
plt.title("Il modello impara: l'errore scende")
plt.xlabel('Epoche')
plt.ylabel('Errore')
plt.legend()
plt.show()

## Passo 5 — Valutare

Il compito in classe: diamo alla rete i dati di test, che non ha mai visto.

In [ ]:
perdita, errore_medio = model_2.evaluate(x_test, y_test, verbose=0)
print(f"Errore medio sui dati mai visti: {errore_medio:.4f}")

predictions = model_2.predict(x_test, verbose=0)

plt.clf()
plt.title('Confronto fra previsioni e valori veri')
plt.plot(x_test, y_test, 'b.', label='Valore vero')
plt.plot(x_test, predictions, 'r.', label='Previsione')
plt.legend()
plt.show()

## Passo 6 — Convertire e comprimere

Da qui in poi siamo in territorio nuovo, e le differenze rispetto alla Parte 1 sono due. Prendetevi
il tempo di leggerle: sono la ragione per cui esiste questa lezione.

### Modifica 2 — tutto a numeri interi, non solo dentro

Il modello che abbiamo adesso ragiona in **virgola mobile**: numeri con la virgola, quattro byte
l'uno. Il microcontrollore preferisce di gran lunga i **numeri interi a 8 bit**, uno solo da -128
a +127. Convertirlo significa "arrotondare con criterio", come abbiamo raccontato all'inizio di
questa lezione.

Nella Parte 1 la conversione si fermava a metà strada: comprimeva i pesi *dentro* il modello, ma
lasciava che i numeri in **entrata e in uscita** restassero con la virgola. Sul computer non fa
differenza. Qui invece sì, per due motivi.

Il primo lo vedete subito: un modello mezzo compresso deve portarsi dietro due operazioni in più,
una che converte l'ingresso all'entrata e una che riconverte l'uscita all'uscita. Il risultato,
misurato, è che **il modello "compresso" viene più grande di quello non compresso**. Nella Parte 1
c'era una cella che stampava il risparmio ottenuto: rifacendo oggi quel conto, il risparmio
verrebbe negativo. Con la conversione completa il risparmio torna, e ve lo faccio vedere.

Il secondo motivo è quello che conta davvero, e riguarda la Lezione 02. Là l'ingresso del modello
non sarà un numero, ma **16.000 numeri**: un secondo di audio. In virgola mobile sono 64.000 byte,
un quarto di tutta la memoria della scheda, buttati in un colpo solo. A numeri interi sono 16.000.
Se imparassimo la ricetta a metà adesso, dovremmo cambiarla fra due lezioni.

Le tre righe che fanno la differenza sono `target_spec.supported_ops`, `inference_input_type` e
`inference_output_type`.

Resta invece uguale alla Parte 1 il `representative_dataset`: il convertitore, prima di decidere
il fattore di scala, vuole **vedere qualche dato vero** per capire in che intervallo si muovono i
numeri. È come far provare l'abito a qualcuno prima di prendere le misure definitive.

### Modifica 3 — la riga che sembra assurda e senza cui non funziona niente

`_experimental_disable_per_channel_quantization_for_dense_layers` è illeggibile, ed è la riga più
importante del notebook.

Quando comprime i pesi, il convertitore può scegliere fra due strade: usare **un solo fattore di
scala per tutto lo strato**, oppure **uno diverso per ogni neurone**. La seconda è più precisa, e
le versioni recenti di TensorFlow la applicano di loro iniziativa agli strati Dense.

Il guaio è che la libreria Arduino con cui lavoreremo è ferma a fine 2023, e a quell'epoca i suoi
strati Dense sapevano leggere **soltanto** i modelli con un unico fattore di scala. Se le date un
modello con un fattore per neurone, non protesta e non dà errore: legge il primo, ignora tutti
gli altri, e comincia a restituire numeri sbagliati con l'aspetto di numeri giusti. Il modello sul
computer funziona benissimo, e lo stesso identico file sulla scheda risponde a caso.

Questa riga dice al convertitore di comportarsi come si faceva una volta. Il modello diventa perfino
un po' più piccolo, e la precisione non ne risente in modo percepibile.

**La regola da portarsi dietro per sempre:** il programma che comprime il modello e la libreria che
lo esegue devono parlare la stessa lingua. Quando i due sono di epoche diverse, controllate sempre
che la scheda dia gli stessi numeri del computer sugli stessi ingressi, prima di dare per buono
qualunque risultato. Nel seguito del libro lo faremo sistematicamente.

In [ ]:
# Prima conversione: nessuna compressione. Serve solo come termine di paragone.
convertitore = tf.lite.TFLiteConverter.from_keras_model(model_2)
modello_non_compresso = convertitore.convert()
open("sine_model.tflite", "wb").write(modello_non_compresso)

# Seconda conversione: compressione completa a numeri interi a 8 bit.
convertitore = tf.lite.TFLiteConverter.from_keras_model(model_2)

# Chiediamo le ottimizzazioni, fra cui la quantizzazione
convertitore.optimizations = [tf.lite.Optimize.DEFAULT]

# Mostriamo al convertitore qualche dato vero, perche' possa scegliere i fattori di scala
def representative_dataset_generator():
    for value in x_train:
        yield [np.array(value, dtype=np.float32, ndmin=2)]

convertitore.representative_dataset = representative_dataset_generator

# --- MODIFICA 2: interi anche in entrata e in uscita, non solo dentro ---
convertitore.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
convertitore.inference_input_type = tf.int8
convertitore.inference_output_type = tf.int8

# --- MODIFICA 3: un solo fattore di scala per strato, come sa leggere la libreria Arduino ---
convertitore._experimental_disable_per_channel_quantization_for_dense_layers = True

modello_compresso = convertitore.convert()
open("sine_model_quantized.tflite", "wb").write(modello_compresso)

import os
dim_base = os.path.getsize("sine_model.tflite")
dim_compresso = os.path.getsize("sine_model_quantized.tflite")
print(f"Modello non compresso : {dim_base} byte")
print(f"Modello compresso     : {dim_compresso} byte")
print(f"Risparmio             : {dim_base - dim_compresso} byte")

### Il controllo che conviene fare sempre

Abbiamo appena arrotondato tutti i numeri del modello. Ha ancora imparato il seno, o l'abbiamo
rovinato? Si scopre in due minuti, e conviene farlo **prima** di passare mezz'ora sull'IDE Arduino.

Notate come si parla al modello compresso: gli si passa un numero intero, non più il valore di x.
La traduzione la facciamo noi, con la **scala** e lo **zero** che il convertitore ha calcolato e
lasciato scritti dentro il file. Sono esattamente gli stessi due numeri, e la stessa identica
formula, che useremo fra poco nello sketch Arduino.

In [ ]:
interprete = tf.lite.Interpreter(model_path="sine_model_quantized.tflite")
interprete.allocate_tensors()

dettagli_in  = interprete.get_input_details()[0]
dettagli_out = interprete.get_output_details()[0]
scala_in,  zero_in  = dettagli_in['quantization']
scala_out, zero_out = dettagli_out['quantization']

print("Ingresso:", dettagli_in['dtype'].__name__, dettagli_in['shape'],
      f" scala={scala_in:.9f} zero={zero_in}")
print("Uscita  :", dettagli_out['dtype'].__name__,
      f" scala={scala_out:.9f} zero={zero_out}")

previsioni_compresse = []
errori = []
for valore_x, valore_y in zip(x_test, y_test):
    # dal valore vero al numero intero (questa e' la formula dello sketch)
    intero = int(round(valore_x / scala_in) + zero_in)
    intero = max(-128, min(127, intero))
    interprete.set_tensor(dettagli_in['index'], np.array([[intero]], dtype=np.int8))
    interprete.invoke()
    # dal numero intero al valore vero (anche questa e' nello sketch)
    grezzo = int(interprete.get_tensor(dettagli_out['index'])[0][0])
    risultato = (grezzo - zero_out) * scala_out
    previsioni_compresse.append(risultato)
    errori.append(abs(risultato - valore_y))

print(f"\nErrore medio PRIMA della compressione: {errore_medio:.4f}")
print(f"Errore medio DOPO  la compressione: {np.mean(errori):.4f}")

plt.clf()
plt.title('Il modello compresso regge il confronto?')
plt.plot(x_test, y_test, 'b.', label='Valore vero')
plt.plot(x_test, predictions, 'r.', label='Modello originale')
plt.plot(x_test, previsioni_compresse, 'gx', label='Modello compresso a 8 bit')
plt.legend()
plt.show()

## Passo 7 — Da file a codice: l'array C

Sulla scheda non c'è un disco su cui appoggiare un file separato: il modello deve diventare
**codice sorgente**, un lungo elenco di byte scritto dentro il programma stesso.

> **Modifica 4 — perché.** Nella Parte 1 questo passaggio si faceva con `!apt-get install xxd` e
> poi `!xxd -i`. Funziona, ma richiede di installare un programma, e produce un array con un nome
> che poi va cambiato a mano. Su Windows le alternative passano da PowerShell, che chi legge da
> Mac o da Linux non ha. Le poche righe qui sotto fanno la stessa identica cosa in Python puro:
> girano ovunque, non installano niente, e scrivono direttamente il nome `g_model` che lo sketch
> si aspetta, con la parola `alignas(8)` che serve al microcontrollore per leggere l'array
> partendo da un indirizzo di memoria buono.

In [ ]:
def scrivi_array_c(file_tflite, file_cpp="model.cpp", nome_array="g_model"):
    dati = open(file_tflite, "rb").read()
    righe = ['#include "model.h"', '', f'alignas(8) const unsigned char {nome_array}[] = {{']
    for i in range(0, len(dati), 12):
        righe.append('  ' + ', '.join(f'0x{b:02x}' for b in dati[i:i+12]) + ',')
    righe.append('};')
    righe.append(f'const int {nome_array}_len = {len(dati)};')
    open(file_cpp, "w").write('\n'.join(righe) + '\n')
    print(f"{file_cpp} scritto: {len(dati)} byte, {len(dati)//12 + 1} righe di numeri")
    return len(dati)

lunghezza = scrivi_array_c("sine_model_quantized.tflite")

# Le prime righe, per vedere che aspetto ha
print()
print(''.join(open("model.cpp").readlines()[:6]))

### Scaricare il file

Cliccate l'icona della cartella nella colonna a sinistra, tasto destro su `model.cpp` →
**Scarica**. Oppure eseguite la cella qui sotto.

In [ ]:
from google.colab import files
files.download('model.cpp')

## Passo 8 — Compilare e caricare

Mettete `model.cpp` nella cartella dello sketch `hello_world_mio`, accanto a
`hello_world_mio.ino` e `model.h`, al posto di quello che c'è già.

Poi, nell'IDE Arduino: **Strumenti → Scheda → Arduino Nano 33 BLE**, **Strumenti → Porta** e
scegliete quella che compare quando collegate la scheda. Premete prima **Verifica** (il segno di
spunta), che compila senza toccare la scheda ed è il modo più veloce per scoprire un errore, e poi
**Carica** (la freccia).

Aprite infine **Strumenti → Monitor Seriale** e sceglete **9600 baud** nel menu in basso a destra.
Dovete vedere scorrere tre colonne — il valore di x, il seno previsto dalla scheda e il seno vero
— e il LED della scheda respirare piano.

Se le prime due colonne si somigliano, avete appena percorso con le vostre mani l'intero tragitto
che dà il titolo a questo libro: da un modello addestrato su Google Colab a un microcontrollore
che lo fa vivere per davvero, senza cavo e senza connessione a internet.

---

## Riepilogo: cosa è cambiato rispetto alla Parte 1, e perché

| Modifica | Motivo |
|---|---|
| `layers.Input(shape=(1,))` come primo strato | Allineamento alle versioni recenti di Keras; il modello risultante è identico. |
| Il rumore riparte da `np.sin(x_values)` invece di sommarsi | Perché la cella si possa rieseguire senza peggiorare i dati a ogni giro. |
| `target_spec.supported_ops`, `inference_input_type`, `inference_output_type` | Per avere numeri interi anche in entrata e in uscita: senza, il modello compresso è più grande di quello non compresso, e nella Lezione 02 l'ingresso occuperebbe 64 KB invece di 16. |
| `_experimental_disable_per_channel_quantization_for_dense_layers` | Perché la libreria Arduino, ferma al 2023, sa leggere un solo fattore di scala per strato. Senza questa riga il modello funziona sul computer e sbaglia sulla scheda **senza dare errore**. |
| L'array C generato in Python invece che con `xxd` | Perché funzioni su qualunque sistema operativo, senza installare niente e senza rinominare l'array a mano. |